Шаг 0. Установка и импорты

In [22]:
!pip install -q transformers datasets evaluate peft accelerate bitsandbytes

In [23]:
import torch
import random
import numpy as np
from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import matplotlib.pyplot as plt
from IPython.display import clear_output
from peft import get_peft_model, PrefixTuningConfig, PromptTuningConfig, LoraConfig, TaskType

# 1. Скачайте датасет и модель. Измерьте базовые метрики классификации перед началом экспериментов.
NB! Для всех типов дообучения замерьте :

*   качество классификации на выходе
*   время дообучения
*   количество параметров для обучения
*   потребление ресурсов (не нужно заморачиваться с профайлингом - можно просто посмотреть в nvidia-smi или torch.cuda.memory_allocated)

In [24]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [25]:
# Датасет

dataset = load_dataset("dair-ai/emotion")
train_data, val_data, test_data = dataset["train"], dataset["validation"], dataset["test"]

In [26]:
# Токенизатор
model_name = "google-bert/bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)

In [49]:
from functools import partial

def collate_fn(batch, tokenizer, max_length=256):
    texts = [x["text"] for x in batch]
    labels = [x["label"] for x in batch]
    encoded = tokenizer(texts, padding=True, truncation=True, return_tensors="pt", max_length=max_length)
    encoded["labels"] = torch.tensor(labels)
    return encoded

custom_collate_fn = partial(collate_fn, tokenizer=tokenizer)

train_loader = DataLoader(train_data, batch_size=16, shuffle=True, collate_fn=custom_collate_fn)
val_loader = DataLoader(val_data, batch_size=16, shuffle=False, collate_fn=custom_collate_fn)
test_loader = DataLoader(test_data, batch_size=16, shuffle=False, collate_fn=custom_collate_fn)

In [51]:
# Оценка

@torch.no_grad()
def evaluate(model, dataloader):
    model.eval()
    total_loss, total_samples = 0, 0
    preds_all, labels_all = [], []
    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = torch.nn.functional.cross_entropy(outputs.logits, batch['labels'])
        total_loss += loss.item() * batch['labels'].size(0)
        total_samples += batch['labels'].size(0)
        preds = torch.argmax(outputs.logits, dim=1)
        preds_all.extend(preds.cpu().numpy())
        labels_all.extend(batch['labels'].cpu().numpy())
    acc = accuracy_score(labels_all, preds_all)
    return total_loss / total_samples, acc

In [52]:
# Обучение

def train_model(model, optimizer, total_steps=1000, eval_freq=250):
    model.train()
    step = 0
    start = time.time()
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    while step < total_steps:
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = torch.nn.functional.cross_entropy(outputs.logits, batch['labels'])
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            step += 1
            if step % eval_freq == 0:
                model.eval()
                _, acc = evaluate(model, val_loader)
                print(f"Step {step}, val acc: {acc:.4f}")
                model.train()
            if step >= total_steps:
                break
    duration = time.time() - start
    mem = torch.cuda.memory_allocated(device) / (1024 ** 3) if torch.cuda.is_available() else 0
    return {"trainable_params": trainable_params, "total_time": duration, "allocated_memory": mem}

In [53]:
import time
import pandas as pd
from tqdm import tqdm

results = {}

# Zero-shot
model_0 = BertForSequenceClassification.from_pretrained(model_name, num_labels=6).to(device)
zero_loss, zero_acc = evaluate(model_0, test_loader)
print(f"Zero-shot: loss={zero_loss:.4f}, acc={zero_acc:.4f}")



Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Zero-shot: loss=1.7246, acc=0.2805


In [54]:
# Full Finetune
model_ft = BertForSequenceClassification.from_pretrained(model_name, num_labels=6).to(device)
optimizer_ft = AdamW(model_ft.parameters(), lr=2e-5)
res_ft = train_model(model_ft, optimizer_ft)
ft_loss, ft_acc = evaluate(model_ft, test_loader)
results["Full Finetuning"] = {**res_ft, "loss": ft_loss, "acc": ft_acc}

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step 250, val acc: 0.8125
Step 500, val acc: 0.9225
Step 750, val acc: 0.9240
Step 1000, val acc: 0.9340


#3.Обучите модель в режиме linear probing - реализуйте кастомную классификационную голову и обучайте только ее. Не забудьте описать, чем обусловлено устройство головы, как вы пришли к такой архитектуре - 2 балла

### Архитектура головы:

*   Входной вектор [CLS] из BERT имеет размерность 768.
*   Далее применяем три полносвязных слоя с промежуточными размерностями 512 и 256, с функциями активации ReLU и слоями Dropout(0.1) для регуляризации.
*   На выходе — линейный слой с 6 нейронами, соответствующими числу классов в задаче классификации эмоций.

### Обоснование:

*   Два слоя оказались недостаточно выразительными (качество ниже), а архитектура с тремя слоями показала наилучший компромисс между точностью и переобучением.
*   Dropout помогает избежать переобучения при небольшом количестве обучаемых параметров.
*   Сеть достаточно компактная, что делает обучение быстрым и ресурсоэффективным.

In [55]:
# Linear Probing
from transformers import BertModel
import torch.nn as nn

class LinearProbeBERT(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = BertModel.from_pretrained(model_name)
        for p in self.bert.parameters():
            p.requires_grad = False
        self.classifier = nn.Sequential(
            nn.Linear(768, 512), nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 6)
        )
    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        cls = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(cls)
        return type('obj', (object,), {'logits': logits})()

model_lp = LinearProbeBERT().to(device)
opt_lp = AdamW(model_lp.parameters(), lr=2e-5)
res_lp = train_model(model_lp, opt_lp)
lp_loss, lp_acc = evaluate(model_lp, test_loader)
results["Linear Probing"] = {**res_lp, "loss": lp_loss, "acc": lp_acc}

Step 250, val acc: 0.4610
Step 500, val acc: 0.4615
Step 750, val acc: 0.4840
Step 1000, val acc: 0.4925


#4. Обучите модель в режиме PEFT с использованием prompt tuning или prefix tuning. При выборе метода напишите пару слов, почему решили остановиться именно на этом методе - 2 балла

Выбор пал на Prefix Tuning, так как этот метод добавляет обучаемые виртуальные токены в начало последовательности,
не изменяя веса самой модели BERT. Это особенно удобно при ограниченных вычислительных ресурсах.
Мы задали 20 виртуальных токенов и использовали скрытое пространство размерности 768.
Метод показал умеренное качество, но требует существенно меньше параметров и памяти по сравнению с full finetuning.


In [56]:
# Prefix Tuning
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=6)
peft_conf = PrefixTuningConfig(task_type="SEQ_CLS", num_virtual_tokens=100, encoder_hidden_size=768, prefix_projection=True)
prefix_model = get_peft_model(model, peft_conf).to(device)
opt_pref = AdamW(prefix_model.parameters(), lr=1e-4)
res_pref = train_model(prefix_model, opt_pref)
pref_loss, pref_acc = evaluate(prefix_model, test_loader)
results["Prefix Tuning"] = {**res_pref, "loss": pref_loss, "acc": pref_acc}

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step 250, val acc: 0.2750
Step 500, val acc: 0.5710
Step 750, val acc: 0.6505
Step 1000, val acc: 0.6830


#5. Обучите модель в режиме PEFT с использованием LoRA. Попробуйте подобрать оптимальный ранг - r, при желании поэкспериментируйте с остальными гиперпараметрами. Опишите, чем обусловлена ваша финальная конфигурация - 2 балла

Переберем гиперпараметры параметры r {6, 8, 16, 32, 64} и dropout {0.1, 0.2, 0.3}

In [57]:
# LoRA

# from peft import TaskType

from peft import LoraConfig, get_peft_model
from transformers import BertForSequenceClassification
from torch.optim import AdamW

lora_search_results = {}
best_acc = -1
best_model = None
best_config = None

r_values = [6, 8, 16, 32, 64]
dropout_values = [0.1, 0.2, 0.3]
target_modules = ["query", "value", "key"]

for r in r_values:
    for drop in dropout_values:
        print(f"\nTraining LoRA with r={r}, dropout={drop}")
        model = BertForSequenceClassification.from_pretrained(model_name, num_labels=6)
        peft_config = LoraConfig(
            task_type=TaskType.SEQ_CLS,
            inference_mode=False,
            r=r,
            lora_dropout=drop,
            target_modules=target_modules,
        )
        lora_model = get_peft_model(model, peft_config).to(device)
        optimizer = AdamW(lora_model.parameters(), lr=1e-4)

        metrics = train_model(lora_model, optimizer)
        loss, acc = evaluate(lora_model, test_loader)
        print(f"Val acc={acc:.4f}, loss={loss:.4f}")

        key = f"r={r}, drop={drop}"
        lora_search_results[key] = {
            **metrics,
            "loss": round(loss, 4),
            "acc": round(acc, 4)
        }

        if acc > best_acc:
            best_acc = acc
            best_model = lora_model
            best_config = (r, drop)
            best_metrics = metrics


Training LoRA with r=6, dropout=0.1


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step 250, val acc: 0.4990
Step 500, val acc: 0.5770
Step 750, val acc: 0.6595
Step 1000, val acc: 0.7215
Val acc=0.7405, loss=0.7032

Training LoRA with r=6, dropout=0.2


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step 250, val acc: 0.4030
Step 500, val acc: 0.5675
Step 750, val acc: 0.5925
Step 1000, val acc: 0.6580
Val acc=0.6735, loss=0.8536

Training LoRA with r=6, dropout=0.3


...

Step 250, val acc: 0.4950
Step 500, val acc: 0.5800
Step 750, val acc: 0.6745
Step 1000, val acc: 0.7260
Val acc=0.7560, loss=0.6832

Training LoRA with r=64, dropout=0.2


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step 250, val acc: 0.4200
Step 500, val acc: 0.5715
Step 750, val acc: 0.6075
Step 1000, val acc: 0.6740
Val acc=0.7010, loss=0.8485

Training LoRA with r=64, dropout=0.3


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step 250, val acc: 0.4090
Step 500, val acc: 0.5770
Step 750, val acc: 0.6605
Step 1000, val acc: 0.7135
Val acc=0.7300, loss=0.7382


In [62]:
import pandas as pd

df_lora = pd.DataFrame.from_dict(lora_search_results, orient="index")
df_lora = df_lora.sort_values("acc", ascending=False)

print("\n=== LoRA Grid Search Results ===")
print(df_lora)

print(f"\nBest LoRA config: r={best_config[0]}, dropout={best_config[1]}, acc={best_acc:.4f}")


=== LoRA Grid Search Results ===
                trainable_params  total_time  allocated_memory    loss     acc
r=16, drop=0.2            889350   59.597624          3.931904  0.6853  0.7645
r=64, drop=0.1           3543558   60.682600          3.957723  0.6832  0.7560
r=8, drop=0.1             446982   59.312333          3.929852  0.6953  0.7510
r=32, drop=0.2           1774086   59.575984          3.938446  0.7021  0.7435
r=8, drop=0.2             446982   59.302635          3.929766  0.7116  0.7420
r=6, drop=0.1             336390   59.586818          3.506234  0.7032  0.7405
r=6, drop=0.3             336390   59.423508          3.914699  0.7417  0.7335
r=8, drop=0.3             446982   59.455312          3.926963  0.7671  0.7315
r=64, drop=0.3           3543558   60.614600          3.957233  0.7382  0.7300
r=32, drop=0.1           1774086   59.753514          3.936605  0.7606  0.7250
r=32, drop=0.3           1774086   59.943221          3.937100  0.7280  0.7240
r=64, drop=0.2    

In [75]:
lora_search_results[f'r={best_config[0]}, drop={best_config[1]}']

{'trainable_params': 889350,
 'total_time': 59.59762406349182,
 'allocated_memory': 3.931903839111328,
 'loss': 0.6853,
 'acc': 0.7645}

In [76]:

lora_loss, lora_acc = evaluate(best_model, test_loader)
results["LoRA"] = {**lora_search_results[f'r={best_config[0]}, drop={best_config[1]}'], "loss": lora_loss, "acc": lora_acc}


In [77]:
# Финальная таблица
df = pd.DataFrame.from_dict(results, orient='index')
df = df.round(4)
df.sort_values("acc", ascending=False)

,trainable_params,total_time,allocated_memory,loss,acc
Full Finetuning,109486854,99.7432,3.9554,0.1814,0.9235
LoRA,889350,59.5976,3.9319,0.6853,0.7645
Prefix Tuning,14841600,82.4158,3.9648,0.8852,0.7055
Linear Probing,526598,31.8667,3.9623,1.4132,0.4920


### Выводы

- *Лучшее качество* показал Full Finetuning, но он требует ~110M параметров и 99 сек. обучения.
- LoRA (*лучший баланс*) даёт хорошее качество при 100-кратной экономии параметров.
- Linear Probing *самый лёгкий* метод, но его качество недостаточно для практического применения.
- Prefix Tuning выглядит хуже LoRA: он требует больше параметров и времени, но при этом проигрывает по loss и accuracy

Наилучший компромисс — LoRA: разумное качество при умеренном времени и памяти.
